# 02 — Capacidad residencial privada frente a población 60+ y dependiente, por alcaldía

**Para qué decisión existe este notebook.** Saber si el faltante de oferta residencial en el oriente, que salió en `01_denue_oferta_cuidado.ipynb` (conteo de establecimientos), se sostiene al medir **capacidad (plazas)** contra **población 60+**, y no contra conteos.
Esto importa para el programa. Si el problema es escasez, un matching no lo resuelve.

**Estado:** exploratorio. Resultado en nivel **probable**, no confirmado (ver sección 5).

**Métrica principal.** Cociente de localización (QL):

$$QL_i = \frac{\text{plazas}_i / \sum \text{plazas}}{\text{población}_i / \sum \text{población}}$$

QL = 1 significa que la alcaldía tiene la parte de la capacidad que le corresponde por su población. QL < 1 significa que está subrepresentada.
Se usa QL porque el archivo de población 60+ viene **en porcentaje del total de la CDMX**, no en personas. Con esta métrica no hace falta inventar un total de población.

In [ ]:
from pathlib import Path
import unicodedata
import pandas as pd
from scipy.stats import spearmanr

DATA_DIR = Path('datos')        # carpeta con los CSV tal como están en el Project
OUT_DIR = Path('resultados'); OUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.width', 160)

F_CAP  = 'Capacidad disponible en residencias permanentes de sector privado para adultos y adultas mayores.csv'
F_DISC = 'Población con discapacidad que necesita ayuda para las actividades de la vida diaria.csv'
F_P60  = 'Población de más de 60 años.csv'
F_PROY = 'Proyección de la población por grupos de edad y alcaldía en porcentaje.csv'
F_PUB  = 'Centros de día para adultos y adultas mayores del sector público.csv'

def leer(f):
    df = pd.read_csv(DATA_DIR / f)
    df.columns = [c.strip() for c in df.columns]   # los CSV traen espacios al inicio del encabezado
    return df

cap, disc, p60, proy, pub = (leer(f) for f in [F_CAP, F_DISC, F_P60, F_PROY, F_PUB])
for n, d in zip(['cap','disc','p60','proy','pub'], [cap, disc, p60, proy, pub]):
    print(f'{n:5s}', d.shape, list(d.columns))

cap   (12, 3) ['Alcaldia', 'Capacidad_total', 'Nomenclartura alcaldia']
disc  (204, 4) ['tipo', 'total', 'alcaldia', 'poblacion']
p60   (102, 3) ['NOM_MUN', 'GRUPO_EDAD', 'PORCENTAJE']
proy  (7616, 6) ['MUN', 'AÑO', 'SEXO', 'NOM_ENT', 'EDAD_QUIN', 'PORCENTAJE']
pub   (33, 3) ['Tipo', 'Municipio', 'Numero_instituciones']


## 1. Llave común de alcaldía

Cada archivo nombra las alcaldías distinto: con o sin acento, en mayúsculas, "Cuajimalpa de Morelos" o "CUAJIMALPA", o claves como `IZTAP`. Todo se normaliza a la clave corta que usan los archivos de discapacidad y capacidad.

In [ ]:
def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower().strip()
    return s.replace(' de morelos', '').replace('.', '')

CLAVE = {'alvaro obregon':'AO','azcapotzalco':'AZC','benito juarez':'BJ','coyoacan':'COY','cuajimalpa':'CUAJ',
         'cuauhtemoc':'CUAU','gustavo a madero':'GAM','iztacalco':'IZTAC','iztapalapa':'IZTAP',
         'la magdalena contreras':'MC','miguel hidalgo':'MH','milpa alta':'MA','tlahuac':'TLAH',
         'tlalpan':'TLAL','venustiano carranza':'VC','xochimilco':'XOCH'}
ALCALDIAS = sorted(CLAVE.values())

p60['clave']  = p60['NOM_MUN'].map(norm).map(CLAVE)
proy['clave'] = proy['MUN'].map(norm).map(CLAVE)
pub['clave']  = pub['Municipio'].map(norm).map(CLAVE)
cap['clave']  = cap['Nomenclartura alcaldia']
# Filas que no son alcaldía: deben ser solo los totales CDMX
print('p60 sin clave:', p60.loc[p60.clave.isna(), 'NOM_MUN'].unique())
print('proy sin clave:', proy.loc[proy.clave.isna(), 'MUN'].unique())
print('pub sin clave:', pub.loc[pub.clave.isna(), 'Municipio'].unique())
print('cap: alcaldías presentes =', cap.clave.nunique(), '| faltan:', sorted(set(ALCALDIAS) - set(cap.clave)))

p60 sin clave: <StringArray>
['Ciudad de México']
Length: 1, dtype: str
proy sin clave: <StringArray>
['CDMX']
Length: 1, dtype: str
pub sin clave: <StringArray>
[]
Length: 0, dtype: str
cap: alcaldías presentes = 12 | faltan: ['CUAJ', 'MA', 'TLAH', 'VC']


## 2. Qué representa cada archivo (fichas)

Lo que se puede verificar **dentro del archivo** se comprueba en código. Lo que no se puede verificar (institución productora, año, licencia) queda como **desconocido**. No se rellena con suposiciones.

In [ ]:
# Chequeo: en el archivo 60+, ¿las alcaldías suman exactamente la fila CDMX? Si sí, el porcentaje es del total de población de la CDMX.
suma = p60[p60.clave.notna()].groupby('GRUPO_EDAD').PORCENTAJE.sum()
cdmx = p60[p60.clave.isna()].set_index('GRUPO_EDAD').PORCENTAJE
chk = pd.DataFrame({'suma_alcaldias': suma, 'fila_CDMX': cdmx}); chk['dif'] = chk.suma_alcaldias - chk.fila_CDMX
print(chk.round(6)); print('60+ como % de la población total CDMX:', round(cdmx.sum(), 2))

              suma_alcaldias  fila_CDMX  dif
GRUPO_EDAD                                  
60 a 64 años        5.097555   5.097555 -0.0
65 a 69 años        3.867353   3.867353 -0.0
70 a 74 años        2.907010   2.907010  0.0
75 a 79 años        1.902281   1.902281  0.0
80 a 84 años        1.275523   1.275523  0.0
85 y más            1.145099   1.145099  0.0
60+ como % de la población total CDMX: 16.19


**Año del archivo 60+.** El archivo no tiene año. Para acotarlo se compara con la proyección, que sí tiene años (2015–2030). En el grupo de 60 a 64 años, se busca el año de la proyección cuya distribución por alcaldía más se parece a la del archivo.

In [ ]:
a = (proy[(proy.EDAD_QUIN == '60 a 64 años') & proy.clave.notna()]
       .groupby(['AÑO', 'clave']).PORCENTAJE.sum().rename('proy').reset_index())
b = p60[(p60.GRUPO_EDAD == '60 a 64 años') & p60.clave.notna()][['clave', 'PORCENTAJE']].rename(columns={'PORCENTAJE': 'p60'})
m = a.merge(b, on='clave')
difs = m.assign(d=(m.proy - m.p60).abs()).groupby('AÑO').d.agg(['max', 'mean']).round(4)
print(difs)
print('Año más cercano:', difs['max'].idxmin(), '— ¿coincidencia exacta?', bool(difs['max'].min() < 1e-6))

         max    mean
AÑO                 
2015  0.1823  0.0526
2016  0.1494  0.0424
2017  0.1203  0.0335
2018  0.0920  0.0258
2019  0.0648  0.0198
2020  0.0389  0.0150
2021  0.0399  0.0124
2022  0.0488  0.0139
2023  0.0569  0.0185
2024  0.0642  0.0230
2025  0.0708  0.0280
2026  0.0771  0.0329
2027  0.0913  0.0378
2028  0.1056  0.0431
2029  0.1199  0.0487
2030  0.1343  0.0547
Año más cercano: 2020 — ¿coincidencia exacta? False


**Lectura:** el archivo 60+ se parece más a **2020**, pero **no coincide exactamente** con ningún año de la proyección (diferencia máxima ≈ 0.04 puntos). Es otra fuente.
Que sea el Censo 2020 es **probable, no confirmado**: la cercanía a 2020 lo sugiere, pero hay que verificarlo con quien descargó el archivo.

In [ ]:
fichas = pd.DataFrame([
  dict(archivo=F_P60, unidad_observacion='alcaldía × grupo quinquenal 60+', grano='16 alcaldías + total CDMX × 6 grupos',
       clave='NOM_MUN + GRUPO_EDAD', medida='% de la población TOTAL de la CDMX (verificado: alcaldías suman fila CDMX)',
       anio='desconocido; más cercano a 2020 contra la proyección, sin coincidencia exacta', productor='desconocido',
       factor_expansion='no aplica / desconocido', observaciones_independientes='no (composicional)'),
  dict(archivo=F_CAP, unidad_observacion='alcaldía', grano='12 alcaldías (faltan VC, TLAH, CUAJ, MA)',
       clave='Nomenclartura alcaldia', medida='plazas en residencias permanentes privadas',
       anio='desconocido', productor='desconocido',
       factor_expansion='no aplica', observaciones_independientes='sí'),
  dict(archivo=F_DISC, unidad_observacion='alcaldía × tipo (Limitación/Discapacidad) × actividad', grano='16 alcaldías + CDMX',
       clave='alcaldia + tipo + poblacion', medida='personas (conteo), TODAS las edades',
       anio='desconocido; la tipología limitación/discapacidad es la del Censo 2020 (probable)', productor='desconocido',
       factor_expansion='desconocido (si es cuestionario ampliado, ya viene expandido)', observaciones_independientes='no: una persona puede estar en varias actividades'),
  dict(archivo=F_PROY, unidad_observacion='alcaldía × año × sexo × grupo quinquenal', grano='2015–2030',
       clave='MUN + AÑO + SEXO + EDAD_QUIN', medida='% de la población total CDMX (suma 200 por año: 100 alcaldías + 100 fila CDMX)',
       anio='2015–2030 (proyección)', productor='desconocido',
       factor_expansion='no aplica', observaciones_independientes='no (composicional)'),
  dict(archivo=F_PUB, unidad_observacion='alcaldía × tipo de institución', grano='33 filas',
       clave='Tipo + Municipio', medida='número de instituciones (NO capacidad)',
       anio='desconocido', productor='desconocido',
       factor_expansion='no aplica', observaciones_independientes='sí'),
])
fichas.to_csv(OUT_DIR / '02_fichas_fuentes.csv', index=False, encoding='utf-8-sig')
fichas[['archivo', 'medida', 'anio']]

,archivo,medida,anio
0,Población de más de 60 años.csv,% de la población TOTAL de la CDMX (verificado...,desconocido; más cercano a 2020 contra la proy...
1,Capacidad disponible en residencias permanente...,plazas en residencias permanentes privadas,desconocido
2,Población con discapacidad que necesita ayuda ...,"personas (conteo), TODAS las edades",desconocido; la tipología limitación/discapaci...
3,Proyección de la población por grupos de edad ...,% de la población total CDMX (suma 200 por año...,2015–2030 (proyección)
4,Centros de día para adultos y adultas mayores ...,número de instituciones (NO capacidad),desconocido


## 3. Denominadores

Se usan tres denominadores para ver si el resultado depende de cuál se elige:
- **60+**: toda la población de 60 años y más.
- **80+**: grupos 80–84 y 85+. Es la población que más probablemente necesita una residencia.
- **ADL**: personas con *discapacidad* para bañarse, vestirse o comer. Es de **todas las edades**, porque el archivo no separa por edad.

In [ ]:
pob = pd.DataFrame(index=ALCALDIAS)
x = p60[p60.clave.notna()]
pob['p60'] = x.groupby('clave').PORCENTAJE.sum()
pob['p80'] = x[x.GRUPO_EDAD.isin(['80 a 84 años', '85 y más'])].groupby('clave').PORCENTAJE.sum()
pob['adl'] = (disc[(disc.tipo == 'Discapacidad') & (disc.poblacion == 'para bañarse, vestirse o comer') & (disc.alcaldia != 'CDMX')]
              .set_index('alcaldia').total)
pob['plazas'] = cap.set_index('clave').Capacidad_total
pob

,p60,p80,adl,plazas
AO,1.328108,0.199024,6653,140.0
AZC,0.853968,0.140511,4220,186.0
BJ,0.948366,0.159013,3786,102.0
COY,1.374514,0.216494,6310,361.0
CUAJ,0.280154,0.035950,1448,NaN
CUAU,1.018562,0.154344,4680,197.0
GAM,2.209232,0.363042,11935,76.0
IZTAC,0.769896,0.130153,4088,14.0
IZTAP,2.845446,0.358830,16076,20.0
MA,0.178318,0.023181,1252,NaN


## 4. Cociente de localización (QL)

Hay 4 alcaldías sin dato de capacidad y no sabemos si eso significa **0 plazas** o **dato faltante**. Por eso se calculan dos escenarios:
- **A — excluidas:** las 4 alcaldías salen del cálculo y las participaciones se calculan solo sobre las 12 que tienen dato.
- **B — cero:** las 4 alcaldías cuentan con 0 plazas y entran en las participaciones.

In [ ]:
def ql(df, den):
    return (df.plazas / df.plazas.sum()) / (df[den] / df[den].sum())

A = pob.dropna(subset=['plazas']).copy()
B = pob.assign(plazas=pob.plazas.fillna(0))
res = pd.DataFrame(index=ALCALDIAS)
for den in ['p60', 'p80', 'adl']:
    res[f'QL_{den}_A'] = ql(A, den)
    res[f'QL_{den}_B'] = ql(B, den)
res['plazas'] = pob.plazas
res['plazas_por_1000_adl'] = pob.plazas / pob.adl * 1000
res = res.sort_values('QL_p60_B').round(2)
res.to_csv(OUT_DIR / '02_indice_capacidad_residencial.csv', encoding='utf-8-sig')
res

,QL_p60_A,QL_p60_B,QL_p80_A,QL_p80_B,QL_adl_A,QL_adl_B,plazas,plazas_por_1000_adl
CUAJ,NaN,0.00,NaN,0.00,NaN,0.00,NaN,NaN
VC,NaN,0.00,NaN,0.00,NaN,0.00,NaN,NaN
TLAH,NaN,0.00,NaN,0.00,NaN,0.00,NaN,NaN
MA,NaN,0.00,NaN,0.00,NaN,0.00,NaN,NaN
IZTAP,0.05,0.05,0.06,0.06,0.04,0.05,20.0,1.24
IZTAC,0.12,0.13,0.11,0.12,0.11,0.13,14.0,3.42
GAM,0.23,0.25,0.21,0.23,0.21,0.24,76.0,6.37
AO,0.69,0.78,0.70,0.78,0.70,0.80,140.0,21.04
BJ,0.71,0.80,0.63,0.71,0.90,1.03,102.0,26.94
CUAU,1.27,1.43,1.26,1.41,1.41,1.60,197.0,42.09


In [ ]:
# ¿El orden de las alcaldías cambia según el denominador? (Escenario A: solo las 12 con dato)
r = res.dropna(subset=['QL_p60_A'])
for a_, b_ in [('QL_p60_A', 'QL_p80_A'), ('QL_p60_A', 'QL_adl_A'), ('QL_p80_A', 'QL_adl_A')]:
    rho, p = spearmanr(r[a_], r[b_]); print(f'{a_} vs {b_}: rho={rho:.2f}  p={p:.3f}  n={len(r)}')

bajo = {c: set(res.index[res[c] < 0.5]) for c in ['QL_p60_A', 'QL_p80_A', 'QL_adl_A']}
print('\nQL < 0.5 en los tres denominadores (A):', sorted(set.intersection(*bajo.values())))
print('Participación de Iztapalapa: 60+ =', round(pob.p60['IZTAP'] / pob.p60.sum() * 100, 1), '%  | plazas (A) =',
      round(pob.plazas['IZTAP'] / pob.plazas.sum() * 100, 1), '%')

QL_p60_A vs QL_p80_A: rho=0.99  p=0.000  n=12
QL_p60_A vs QL_adl_A: rho=0.97  p=0.000  n=12
QL_p80_A vs QL_adl_A: rho=0.94  p=0.000  n=12

QL < 0.5 en los tres denominadores (A): ['GAM', 'IZTAC', 'IZTAP']
Participación de Iztapalapa: 60+ = 17.6 %  | plazas (A) = 0.9 %


### ¿La oferta pública compensa?

El archivo público trae **número de instituciones, no plazas**. Además, la mayoría son *clubes*, que no son cuidado residencial.
Aquí solo se cuentan albergues, residencias de día y residencias permanentes, para ver si el sector público cubre donde falta oferta privada. Es indicativo, no es una medida de capacidad.

In [ ]:
pub_res = (pub[pub.Tipo.isin(['ALBERGUE', 'RESIDENCIA DE DIA', 'RESIDENCIAS PERMANENTES'])]
           .groupby(['clave', 'Tipo']).Numero_instituciones.sum().unstack(fill_value=0))
comp = res[['QL_p60_B', 'plazas']].join(pub_res).fillna(0)
comp['pub_no_club_total'] = comp[[c for c in pub_res.columns]].sum(axis=1)
comp.sort_values('QL_p60_B')

,QL_p60_B,plazas,ALBERGUE,RESIDENCIA DE DIA,RESIDENCIAS PERMANENTES,pub_no_club_total
CUAJ,0.00,0.0,0.0,0.0,0.0,0.0
VC,0.00,0.0,0.0,1.0,0.0,1.0
TLAH,0.00,0.0,0.0,0.0,0.0,0.0
MA,0.00,0.0,0.0,0.0,0.0,0.0
IZTAP,0.05,20.0,1.0,0.0,0.0,1.0
IZTAC,0.13,14.0,0.0,0.0,0.0,0.0
GAM,0.25,76.0,1.0,2.0,2.0,5.0
AO,0.78,140.0,3.0,2.0,0.0,5.0
BJ,0.80,102.0,7.0,2.0,0.0,9.0
CUAU,1.43,197.0,0.0,0.0,0.0,0.0


## 5. Qué permite afirmar este resultado y qué no

Ver las tablas de arriba. Los números se leen de `resultados/02_indice_capacidad_residencial.csv`, no se copian a mano al documento.

**Permite afirmar (probable):**
- La capacidad residencial *privada* está muy por debajo de lo proporcional en Iztapalapa (QL ≈ 0.05: tiene cerca del 18% de la población 60+ y menos del 1% de las plazas), Iztacalco y Gustavo A. Madero. Esto se cumple con los tres denominadores y en los dos escenarios.
- El resultado no depende de usar 60+ o 80+. Cambia poco con el denominador ADL.
- En Iztapalapa e Iztacalco el sector público tampoco compensa: tienen 1 y 0 instituciones públicas que no son clubes. Gustavo A. Madero sí tiene 5, incluidas las únicas 2 residencias permanentes públicas del archivo. Pero son conteos de instituciones, no plazas, así que su faltante residencial es menos claro.

**No permite afirmar:**
- Que exista **demanda no atendida** de residencias. Puede haber pocas plazas privadas porque la gente no las puede pagar o porque no las quiere. La ENASIC 2022 (nacional) muestra rechazo cultural a las residencias (ver `enasic_2022_cifras_presentacion.csv`).
- Que la gente de esas alcaldías no tenga acceso a plazas. Las plazas de Miguel Hidalgo o Tlalpan no son exclusivas de sus residentes.
- Nada sobre capacidad **pública** ni sobre VC, TLAH, CUAJ y MA en el escenario A.
- Tendencias en el tiempo: hay un solo corte y su año es desconocido.

**Pendiente para pasar a confirmado:** productor, año y método de captura del CSV de capacidad; saber si las 4 alcaldías ausentes valen 0 o son dato faltante; verificar que el archivo 60+ sea del Censo 2020.